In [ ]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2

# import sys
# from awsglue.transforms import *
# from awsglue.utils import getResolvedOptions
# from pyspark.context import SparkContext
# from awsglue.context import GlueContext
# from awsglue.job import Job
  
# sc = SparkContext.getOrCreate()
# glueContext = GlueContext(sc)
# spark = glueContext.spark_session
# job = Job(glueContext)

In [2]:
%stop_session

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
There is no current session.


In [4]:
%%configure
{
  "--datalake-formats": "delta",
  "--conf": "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore",
  "--enable-glue-datacatalog": "true"
}

The following configurations have been updated: {'--datalake-formats': 'delta', '--conf': 'spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore', '--enable-glue-datacatalog': 'true'}


In [58]:
#Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import *
from delta.tables import DeltaTable
from datetime import *
from pyspark.sql.types import *
from pyspark.sql.utils import AnalysisException

BASE = "s3://data-lake-case-hotmart"

# D-1 (pode fixar uma data no UAT, ex: "2023-03-12")
RUN_DATE = (date.today() - timedelta(days=1)).isoformat() #D-1 troque para a data que deseja navegar
# uat : RUN_DATE = "2023-03-31"   # teste também: "2023-01-23", "2023-02-28", "2023-07-15"

PATH_BRONZE_PURCHASE = f"{BASE}/bronze/purchase"
PATH_BRONZE_PRODUCT_ITEM     = f"{BASE}/bronze/product_item"
PATH_BRONZE_EXTRA_INFO    = f"{BASE}/bronze/purchase_extra_info"

PATH_SILVER_PURCHASE = f"{BASE}/silver/purchase_daily"
PATH_SILVER_PRODUCT_ITEM     = f"{BASE}/silver/product_item_daily"
PATH_SILVER_EXTRA_INFO    = f"{BASE}/silver/purchase_extra_info_daily"


PATH_GOLD            = f"{BASE}/gold/gmv_daily"
PATH_GOLD_GMV        = f"{BASE}/gold/gmv_daily_by_subsidiary"



## BRONZE — purchase (eventos CDC, colunas das prints)

### UAT: como no repo, dados mockados “hard coded”.
### Bronze é append-only.

In [18]:

# Criar DataFrame diretamente (dados do enunciado - Exercício 2)
df_purchase_bronze = spark.createDataFrame([
    ("2023-01-20 22:00:00", "2023-01-20", 55, 15947, 5, "2023-01-20", "2023-01-20", 852852),
    ("2023-01-26 00:01:00", "2023-01-26", 56, 369798, 746520, "2023-01-25", None, 963963),
    ("2023-02-05 10:00:00", "2023-02-05", 55, 160001, 5, "2023-01-20", "2023-01-20", 852852),
    ("2023-02-26 03:00:00", "2023-02-26", 69, 160001, 18, "2023-01-26", "2023-02-28", 96967),
    ("2023-07-15 09:00:00", "2023-07-15", 55, 160001, 5, "2023-01-20", "2023-03-01", 852852)
], schema=[
    "transaction_datetime",
    "transaction_date",
    "purchase_id",
    "buyer_id",
    "prod_item_id",
    "order_date",
    "release_date",
    "producer_id"
])

# Tipos corretos
df_purchase_bronze = df_purchase_bronze \
    .withColumn("transaction_datetime", col("transaction_datetime").cast("timestamp")) \
    .withColumn("transaction_date", col("transaction_date").cast("date")) \
    .withColumn("purchase_id", col("purchase_id").cast("bigint")) \
    .withColumn("buyer_id", col("buyer_id").cast("bigint")) \
    .withColumn("prod_item_id", col("prod_item_id").cast("bigint")) \
    .withColumn("producer_id", col("producer_id").cast("bigint")) \
    .withColumn("order_date", col("order_date").cast("date")) \
    .withColumn("release_date", col("release_date").cast("date"))

# Controle de rastreabilidade
df_purchase_bronze = df_purchase_bronze.withColumn(
    "ingestion_date",
    to_utc_timestamp(current_timestamp(), "UTC")
)


# Persistir Bronze em DELTA (append-only)
df_purchase_bronze.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("transaction_date") \
    .save(PATH_BRONZE_PURCHASE)

In [19]:

# Criar view temporária
df_purchase_bronze.createOrReplaceTempView("vw_bronze_purchase")

spark.sql("SELECT * FROM vw_bronze_purchase").show()

+--------------------+----------------+-----------+--------+------------+----------+------------+-----------+--------------------+
|transaction_datetime|transaction_date|purchase_id|buyer_id|prod_item_id|order_date|release_date|producer_id|      ingestion_date|
+--------------------+----------------+-----------+--------+------------+----------+------------+-----------+--------------------+
| 2023-01-20 22:00:00|      2023-01-20|         55|   15947|           5|2023-01-20|  2023-01-20|     852852|2026-02-27 10:24:...|
| 2023-01-26 00:01:00|      2023-01-26|         56|  369798|      746520|2023-01-25|        null|     963963|2026-02-27 10:24:...|
| 2023-02-05 10:00:00|      2023-02-05|         55|  160001|           5|2023-01-20|  2023-01-20|     852852|2026-02-27 10:24:...|
| 2023-02-26 03:00:00|      2023-02-26|         69|  160001|          18|2023-01-26|  2023-02-28|      96967|2026-02-27 10:24:...|
| 2023-07-15 09:00:00|      2023-07-15|         55|  160001|           5|2023-01-20

## BRONZE — product_item (eventos CDC, colunas das prints)

In [21]:
# Criar DataFrame diretamente (dados do enunciado - Exercício 2)
df_product_item_bronze = spark.createDataFrame([
    ("2023-01-20 22:02:00", "2023-01-20", 55, 696969, 10, 50.00),
    ("2023-01-25 23:59:59", "2023-01-25", 56, 808080, 120, 2400.00),
    ("2023-02-26 03:00:00", "2023-02-26", 69, 373737, 2, 2000.00),
    ("2023-07-12 09:00:00", "2023-07-12", 55, 696969, 10, 55.00)
], schema=[
    "transaction_datetime",
    "transaction_date",
    "purchase_id",
    "product_id",
    "item_quantity",
    "purchase_value"
])

df_product_item_bronze = df_product_item_bronze \
    .withColumn("transaction_datetime", col("transaction_datetime").cast("timestamp")) \
    .withColumn("transaction_date", col("transaction_date").cast("date")) \
    .withColumn("purchase_id", col("purchase_id").cast("bigint")) \
    .withColumn("product_id", col("product_id").cast("bigint")) \
    .withColumn("item_quantity", col("item_quantity").cast("int")) \
    .withColumn("purchase_value", col("purchase_value").cast(DecimalType(18, 2)))

df_product_item_bronze = df_product_item_bronze.withColumn(
    "ingestion_date",
    to_utc_timestamp(current_timestamp(), "UTC")
)

# CELL 3
# Persistir Bronze em DELTA (append-only)
df_product_item_bronze.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("transaction_date") \
    .save(PATH_BRONZE_PRODUCT_ITEM)

NameError: name 'PATH_BRONZE_PRODUCT_ITEM' is not defined


In [22]:
# Criar view temporária
df_product_item_bronze.createOrReplaceTempView("vw_product_item_bronze")

spark.sql("SELECT * FROM vw_product_item_bronze").show()

+--------------------+----------------+-----------+----------+-------------+--------------+--------------------+
|transaction_datetime|transaction_date|purchase_id|product_id|item_quantity|purchase_value|      ingestion_date|
+--------------------+----------------+-----------+----------+-------------+--------------+--------------------+
| 2023-01-20 22:02:00|      2023-01-20|         55|    696969|           10|         50.00|2026-02-27 10:28:...|
| 2023-01-25 23:59:59|      2023-01-25|         56|    808080|          120|       2400.00|2026-02-27 10:28:...|
| 2023-02-26 03:00:00|      2023-02-26|         69|    373737|            2|       2000.00|2026-02-27 10:28:...|
| 2023-07-12 09:00:00|      2023-07-12|         55|    696969|           10|         55.00|2026-02-27 10:28:...|
+--------------------+----------------+-----------+----------+-------------+--------------+--------------------+


## BRONZE — purchase_extra_info (eventos CDC)

In [25]:
# Criar DataFrame diretamente (dados do enunciado - Exercício 2)
df_purchase_extra_info_bronze = spark.createDataFrame([
    ("2023-01-23 00:05:00", "2023-01-23", 55, "nacional"),
    ("2023-01-25 23:59:59", "2023-01-25", 56, "internacional"),
    ("2023-02-28 01:10:00", "2023-02-28", 69, "nacional"),
    ("2023-03-12 07:00:00", "2023-03-12", 69, "internacional")
], schema=[
    "transaction_datetime",
    "transaction_date",
    "purchase_id",
    "subsidiary"
])

df_purchase_extra_info_bronze = df_purchase_extra_info_bronze \
    .withColumn("transaction_datetime", col("transaction_datetime").cast("timestamp")) \
    .withColumn("transaction_date", col("transaction_date").cast("date")) \
    .withColumn("purchase_id", col("purchase_id").cast("bigint")) \
    .withColumn("subsidiary", col("subsidiary").cast("string"))

df_purchase_extra_info_bronze = df_purchase_extra_info_bronze.withColumn(
    "ingestion_date",
    to_utc_timestamp(current_timestamp(), "UTC")
)

# CELL 3
# Persistir Bronze em DELTA (append-only)
df_purchase_extra_info_bronze.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("transaction_date") \
    .save(PATH_BRONZE_EXTRA_INFO)

In [26]:
# view para query teste
df_purchase_extra_info_bronze.createOrReplaceTempView("vw_purchase_extra_info_bronze")

spark.sql("SELECT * FROM vw_purchase_extra_info_bronze").show(truncate = False)

+--------------------+----------------+-----------+-------------+-----------------------+
|transaction_datetime|transaction_date|purchase_id|subsidiary   |ingestion_date         |
+--------------------+----------------+-----------+-------------+-----------------------+
|2023-01-23 00:05:00 |2023-01-23      |55         |nacional     |2026-02-27 10:29:44.382|
|2023-01-25 23:59:59 |2023-01-25      |56         |internacional|2026-02-27 10:29:44.382|
|2023-02-28 01:10:00 |2023-02-28      |69         |nacional     |2026-02-27 10:29:44.382|
|2023-03-12 07:00:00 |2023-03-12      |69         |internacional|2026-02-27 10:29:44.382|
+--------------------+----------------+-----------+-------------+-----------------------+


# Silver

## Silver_Purchase

In [ ]:
# CELL 2
# SILVER_PATH = "data_lake/silver/purchase"
# BRONZE_PATH = "data_lake/bronze/purchase"

# 1) Ler Silver atual (se existir)
try:
    df_silver_current = spark.read.format("delta").load(PATH_SILVER_PURCHASE)
except AnalysisException:
    df_silver_current = None

# 2) Última ingestion processada
last_ingestion = (
    df_silver_current.agg(max("bronze_ingestion_date").alias("max_date")).first()["max_date"]
) if df_silver_current is not None else None

# 3) Ler Bronze (incremental ou full)
df_bronze = spark.read.format("delta").load(PATH_BRONZE_PURCHASE)

df_bronze_incremental = (
    df_bronze.filter(col("ingestion_date") > last_ingestion)
    if last_ingestion
    else df_bronze
)

# 4) Transformações Bronze → Silver
df_bronze_ready = (
    df_bronze_incremental
    .withColumn("transaction_status", when(col("release_date").isNotNull(), "Succesfull").otherwise("Failed"))
    .withColumn("line_created_at", to_utc_timestamp(current_timestamp(), "UTC"))
    .withColumnRenamed("ingestion_date", "bronze_ingestion_date")
)

# 5) Union Silver + Bronze incremental
if df_silver_current is not None:
    cols_drop = [c for c in ["rn", "rk", "is_latest", "current_snapshot"] if c in df_silver_current.columns]
    df_union = df_silver_current.drop(*cols_drop).unionByName(df_bronze_ready)
else:
    df_union = df_bronze_ready

# 6) Deduplicação por evento (CDC) - mesma compra pode ser reenviada
event_window = Window.partitionBy("purchase_id", "transaction_datetime").orderBy(col("bronze_ingestion_date").desc())

df_silver_dedup = (
    df_union
    .withColumn("rn", row_number().over(event_window))
    .filter(col("rn") == 1)
    .drop("rn")
)

# 7) Registro corrente por purchase_id (renomeado)
purchase_window = Window.partitionBy("purchase_id").orderBy(col("transaction_datetime").desc())

df_silver_final = (
    df_silver_dedup
    .withColumn("rk", row_number().over(purchase_window))
    .withColumn("is_current_record", col("rk") == 1)   
    .drop("rk")
)

# 8) Escrita final (rebuild) - DELTA
df_silver_final.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("transaction_date") \
    .save(PATH_SILVER_PURCHASE)

In [49]:

df_silver_final.createOrReplaceTempView("vw_purchase_silver")
spark.sql("SELECT * FROM vw_purchase_silver ORDER BY purchase_id, transaction_datetime").show()

+--------------------+----------------+-----------+--------+------------+----------+------------+-----------+---------------------+------------------+--------------------+-----------------+
|transaction_datetime|transaction_date|purchase_id|buyer_id|prod_item_id|order_date|release_date|producer_id|bronze_ingestion_date|transaction_status|     line_created_at|is_current_record|
+--------------------+----------------+-----------+--------+------------+----------+------------+-----------+---------------------+------------------+--------------------+-----------------+
| 2023-01-20 22:00:00|      2023-01-20|         55|   15947|           5|2023-01-20|  2023-01-20|     852852| 2026-02-27 10:24:...|        Succesfull|2026-02-27 10:40:...|            false|
| 2023-02-05 10:00:00|      2023-02-05|         55|  160001|           5|2023-01-20|  2023-01-20|     852852| 2026-02-27 10:24:...|        Succesfull|2026-02-27 10:40:...|            false|
| 2023-07-15 09:00:00|      2023-07-15|         55

### Silver_Product-Product_Item

In [ ]:
# SILVER_PATH = "data_lake/silver/product_item"
# BRONZE_PATH = "data_lake/bronze/product_item"
# PURCHASE_SILVER_PATH = "data_lake/silver/purchase"  # para enriquecer com prod_item_id/partition (diagrama)

# 1) Ler Silver atual (se existir)
try:
    df_silver_current = spark.read.format("delta").load(PATH_SILVER_PRODUCT_ITEM)
except AnalysisException:
    df_silver_current = None

# 2) Última ingestion processada
last_ingestion = (
    df_silver_current.agg(max("bronze_ingestion_date").alias("max_date")).first()["max_date"]
) if df_silver_current is not None else None

# 3) Ler Bronze (incremental ou full)
df_bronze = spark.read.format("delta").load(PATH_BRONZE_PRODUCT_ITEM)
df_bronze_incremental = (
    df_bronze.filter(col("ingestion_date") > last_ingestion)
    if last_ingestion
    else df_bronze
)

# 4) Transformações Bronze → Silver
df_bronze_ready = (
    df_bronze_incremental
    .withColumn("line_created_at", to_utc_timestamp(current_timestamp(), "UTC"))
    .withColumnRenamed("ingestion_date", "bronze_ingestion_date")
)

# 4.1) Enriquecimento para aderir ao diagrama (prod_item_id, prod_item_partition)
try:
    df_purchase_silver = spark.read.format("delta").load(PATH_SILVER_PURCHASE)
    df_purchase_current = df_purchase_silver.filter(col("is_current_record") == True) \
        .select("purchase_id", "prod_item_id", "prod_item_partition", "purchase_partition")
    df_bronze_ready = df_bronze_ready.join(df_purchase_current, "purchase_id", "left")
except AnalysisException:
    pass

# 5) Union Silver + Bronze incremental
if df_silver_current is not None:
    cols_drop = [c for c in ["rn", "rk", "is_latest", "current_snapshot"] if c in df_silver_current.columns]
    df_union = df_silver_current.drop(*cols_drop).unionByName(df_bronze_ready, allowMissingColumns=True)
else:
    df_union = df_bronze_ready

# 6) Deduplicação por evento (CDC)
event_window = Window.partitionBy("purchase_id", "product_id", "transaction_datetime").orderBy(col("bronze_ingestion_date").desc())

df_dedup = (
    df_union
    .withColumn("rn", row_number().over(event_window))
    .filter(col("rn") == 1)
    .drop("rn")
)

# 7) Registro corrente por (purchase_id, product_id) (renomeado)
item_window = Window.partitionBy("purchase_id", "product_id").orderBy(col("transaction_datetime").desc())

df_silver_product_item_final = (
    df_dedup
    .withColumn("rk", row_number().over(item_window))
    .withColumn("is_current_record", col("rk") == 1)   
    .drop("rk")
)

# 8) Escrita final (rebuild) - DELTA
df_silver_product_item_final.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("transaction_date") \
    .save(PATH_SILVER_PRODUCT_ITEM)

In [53]:

df_silver_product_item_final.createOrReplaceTempView("vw_product_item_silver")
spark.sql("SELECT * FROM vw_product_item_silver ORDER BY purchase_id, product_id, transaction_datetime").show()

+--------------------+----------------+-----------+----------+-------------+--------------+---------------------+--------------------+-----------------+
|transaction_datetime|transaction_date|purchase_id|product_id|item_quantity|purchase_value|bronze_ingestion_date|     line_created_at|is_current_record|
+--------------------+----------------+-----------+----------+-------------+--------------+---------------------+--------------------+-----------------+
| 2023-01-20 22:02:00|      2023-01-20|         55|    696969|           10|          50.0| 2026-02-27 09:55:...|2026-02-27 10:43:...|            false|
| 2023-07-12 09:00:00|      2023-07-12|         55|    696969|           10|          55.0| 2026-02-27 09:55:...|2026-02-27 10:43:...|             true|
| 2023-01-25 23:59:59|      2023-01-25|         56|    808080|          120|        2400.0| 2026-02-27 09:55:...|2026-02-27 10:43:...|             true|
| 2023-02-26 03:00:00|      2023-02-26|         69|    373737|            2|      

In [ ]:
# CELL 2
# SILVER_PATH = "data_lake/silver/purchase_extra_info"
# BRONZE_PATH = "data_lake/bronze/purchase_extra_info"
# PURCHASE_SILVER_PATH = "data_lake/silver/purchase"  # para enriquecer purchase_partition (diagrama)

# 1) Ler Silver atual (se existir)
try:
    df_silver_current = spark.read.format("delta").load(PATH_SILVER_EXTRA_INFO)
except AnalysisException:
    df_silver_current = None

# 2) Última ingestion processada
last_ingestion = (
    df_silver_current.agg(max("bronze_ingestion_date").alias("max_date")).first()["max_date"]
) if df_silver_current is not None else None

# 3) Ler Bronze (incremental ou full)
df_bronze = spark.read.format("delta").load(PATH_BRONZE_EXTRA_INFO)

df_bronze_incremental = (
    df_bronze.filter(col("ingestion_date") > last_ingestion)
    if last_ingestion
    else df_bronze
)

# 4) Transformações Bronze → Silver
df_bronze_ready = (
    df_bronze_incremental
    .withColumn("line_created_at", to_utc_timestamp(current_timestamp(), "UTC"))
    .withColumnRenamed("ingestion_date", "bronze_ingestion_date")
)

# 4.1) Enriquecer purchase_partition (diagrama) a partir do purchase corrente, se existir
try:
    df_purchase_silver = spark.read.format("delta").load(PATH_SILVER_PURCHASE)
    df_purchase_current = df_purchase_silver.filter(col("is_current_record") == True) \
        .select("purchase_id", "purchase_partition")
    df_bronze_ready = df_bronze_ready.join(df_purchase_current, "purchase_id", "left")
except AnalysisException:
    pass

# 5) Union Silver + Bronze incremental
if df_silver_current is not None:
    cols_drop = [c for c in ["rn", "rk", "is_latest", "current_snapshot"] if c in df_silver_current.columns]
    df_union = df_silver_current.drop(*cols_drop).unionByName(df_bronze_ready, allowMissingColumns=True)
else:
    df_union = df_bronze_ready

# 6) Deduplicação por evento (CDC)
event_window = Window.partitionBy("purchase_id", "transaction_datetime").orderBy(col("bronze_ingestion_date").desc())

df_dedup = (
    df_union
    .withColumn("rn", row_number().over(event_window))
    .filter(col("rn") == 1)
    .drop("rn")
)

# 7) Registro corrente por purchase_id (renomeado)
purchase_window = Window.partitionBy("purchase_id").orderBy(col("transaction_datetime").desc())

df_silver_extra_info_final = (
    df_dedup
    .withColumn("rk", row_number().over(purchase_window))
    .withColumn("is_current_record", col("rk") == 1)  
    .drop("rk")
)

# 8) Escrita final (rebuild) - DELTA
df_silver_extra_info_final.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("transaction_date") \
    .save(PATH_SILVER_EXTRA_INFO)

In [55]:
df_silver_extra_info_final.createOrReplaceTempView("vw_purchase_extra_info_silver")
spark.sql("SELECT * FROM vw_purchase_extra_info_silver ORDER BY purchase_id, transaction_datetime").show()

+--------------------+----------------+-----------+-------------+---------------------+--------------------+-----------------+
|transaction_datetime|transaction_date|purchase_id|   subsidiary|bronze_ingestion_date|     line_created_at|Is_current_record|
+--------------------+----------------+-----------+-------------+---------------------+--------------------+-----------------+
| 2023-01-23 00:05:00|      2023-01-23|         55|     nacional| 2026-02-27 10:29:...|2026-02-27 10:48:...|             true|
| 2023-01-25 23:59:59|      2023-01-25|         56|internacional| 2026-02-27 10:29:...|2026-02-27 10:48:...|             true|
| 2023-02-28 01:10:00|      2023-02-28|         69|     nacional| 2026-02-27 10:29:...|2026-02-27 10:48:...|            false|
| 2023-03-12 07:00:00|      2023-03-12|         69|internacional| 2026-02-27 10:29:...|2026-02-27 10:48:...|             true|
+--------------------+----------------+-----------+-------------+---------------------+--------------------+---

In [ ]:
# CELL 2
# Ler Silvers (DELTA)
df_purchase_silver = spark.read.format("delta").load(PATH_SILVER_PURCHASE)
df_product_item_silver = spark.read.format("delta").load(PATH_SILVER_PRODUCT_ITEM)
df_purchase_extra_info_silver = spark.read.format("delta").load(PATH_SILVER_EXTRA_INFO)

# snapshot_date = D-1
df_snapshot_date = spark.sql("SELECT date_sub(current_date(), 1) AS snapshot_date").collect()[0]["snapshot_date"]
print("snapshot_date (D-1):", df_snapshot_date)

# CELL 3
# -------------------------------------------------------------
# GOLD GVM (nível de compra) - foto "as-of" snapshot_date
# Para garantir reprodutibilidade e imutabilidade:
#   sempre escolhemos o último estado com transaction_date <= snapshot_date.
# -------------------------------------------------------------

# Purchase as-of
w_purchase = Window.partitionBy("purchase_id").orderBy(
    col("transaction_date").desc(),
    col("transaction_datetime").desc()
)
df_purchase_asof = (
    df_purchase_silver
    .filter(col("transaction_date") <= lit(df_snapshot_date))
    .withColumn("rn", row_number().over(w_purchase))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Product item as-of (por purchase_id + product_id)
w_item = Window.partitionBy("purchase_id","product_id").orderBy(
    col("transaction_date").desc(),
    col("transaction_datetime").desc()
)
df_item_asof = (
    df_product_item_silver
    .filter(col("transaction_date") <= lit(df_snapshot_date))
    .withColumn("rn", row_number().over(w_item))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Extra info as-of (por purchase_id)
w_extra = Window.partitionBy("purchase_id").orderBy(
    col("transaction_date").desc(),
    col("transaction_datetime").desc()
)
df_extra_asof = (
    df_purchase_extra_info_silver
    .filter(col("transaction_date") <= lit(df_snapshot_date))
    .withColumn("rn", row_number().over(w_extra))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Compras pagas (GMV considera somente release_date preenchida)
df_purchase_asof = df_purchase_asof.filter(col("transaction_status") == "Succesfull")

# Join assíncrono:
# - se item ainda não chegou, purchase_value/item_quantity ficam null e a compra não entra no GMV
# - se extra ainda não chegou, subsidiary fica null (vamos tratar como UNKNOWN na agregação)
df_new_gvm = (
    df_purchase_asof.alias("a")
    .join(df_item_asof.alias("b"), col("a.purchase_id") == col("b.purchase_id"), "left")
    .join(df_extra_asof.alias("c"), col("a.purchase_id") == col("c.purchase_id"), "left")
    .select(
        col("a.transaction_datetime"),
        col("a.purchase_id"),
        col("a.buyer_id"),
        col("a.prod_item_id"),
        col("a.order_date"),
        col("a.release_date"),
        col("a.producer_id"),
        col("b.product_id"),
        col("b.item_quantity"),
        col("b.purchase_value"),
        col("c.subsidiary"),
        current_timestamp().alias("snapshot_datetime"),
        lit(df_snapshot_date).cast("date").alias("snapshot_date"),
        col("a.transaction_date")
    )
)

df_new_gvm.show(truncate=False)

# CELL 4
# -------------------------------------------------------------
# Persistência do GOLD GVM (DELTA) - idempotente por snapshot_date
#
#  - lê o gold existente (se houver)
#  - union + dedup do snapshot do dia
#  - recalcula is_current_snapshot com base no max(snapshot_date)
#  - overwrite (mantendo histórico)
# -------------------------------------------------------------
try:
    df_gold_existing = spark.read.format("delta").load(PATH_GOLD)
    # compatibilidade com versões antigas: remove current_snapshot se existir
    if "current_snapshot" in df_gold_existing.columns:
        df_gold_existing = df_gold_existing.drop("current_snapshot")
    df_gold_union = df_gold_existing.unionByName(df_new_gvm, allowMissingColumns=True)
except AnalysisException:
    df_gold_union = df_new_gvm

w_snap = Window.partitionBy("purchase_id","snapshot_date").orderBy(col("snapshot_datetime").desc())
df_gold_dedup = (
    df_gold_union
    .withColumn("rn", row_number().over(w_snap))
    .filter(col("rn") == 1)
    .drop("rn")
)

max_snapshot = df_gold_dedup.select(max("snapshot_date").alias("mx")).collect()[0]["mx"]
df_gold_final = df_gold_dedup.withColumn("is_current_snapshot", col("snapshot_date") == lit(max_snapshot))

df_gold_final.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("snapshot_date","transaction_date") \
    .save(PATH_GOLD)

df_gold_final.createOrReplaceTempView("gvm_gold")
spark.sql("SELECT * FROM gvm_gold WHERE is_current_snapshot = true ORDER BY purchase_id").show(truncate=False)

# CELL 5
# -------------------------------------------------------------
# DATASET FINAL (entregável): GMV diário por subsidiária
# - regra GMV: sum(item_quantity * purchase_value)
# - particiona por transaction_date (usamos snapshot_date para garantir D-1)
# - is_current_snapshot facilita consumo (usuário não precisa de subquery de MAX)
# -------------------------------------------------------------
df_gmv_daily = spark.sql("""
    SELECT
        snapshot_date,
        release_date,
        COALESCE(subsidiary, 'UNKNOWN') AS subsidiary,
        SUM(item_quantity * purchase_value) AS gmv_value,
        COUNT(DISTINCT purchase_id) AS transaction_count,
        is_current_snapshot,
        snapshot_date AS transaction_date
    FROM gvm_gold
    GROUP BY 1,2,3,6,7
""")

df_gmv_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("transaction_date") \
    .save(PATH_GOLD_GMV)

df_gmv_daily.show(truncate=False)

snapshot_date (D-1): 2026-02-26
+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+-----------------------+-------------+----------------+
|transaction_datetime|purchase_id|buyer_id|prod_item_id|order_date|release_date|producer_id|product_id|item_quantity|purchase_value|subsidiary   |snapshot_datetime      |snapshot_date|transaction_date|
+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+-----------------------+-------------+----------------+
|2023-07-15 09:00:00 |55         |160001  |5           |2023-01-20|2023-03-01  |852852     |696969    |10           |55.0          |nacional     |2026-02-27 10:57:10.769|2026-02-26   |2023-07-15      |
|2023-02-26 03:00:00 |69         |160001  |18          |2023-01-26|2023-02-28  |96967      |373737    |2            |2000.0        |internacional|2026-02-27 10:

## quallity

In [66]:
from pyspark.sql.functions import col, lit, sum as spark_sum, countDistinct, current_timestamp

DQ_PATH = "s3://data-lake-case-hotmart/gold/dq_gmv_run_metrics"
QUARANTINE_PATH = "s3://data-lake-case-hotmart/gold/quarantine_missing_items"

# Base "as-of" corrente do snapshot (mesma fonte do gvm_gold)
df_gvm_snapshot = spark.table("gvm_gold").where(col("snapshot_date") == lit(df_snapshot_date))

# 1) Quarentena: compras faturadas sem item (não entram no GMV)
df_quarantine_missing_item = (
    df_gvm_snapshot
    .where(col("release_date").isNotNull())
    .where(col("product_id").isNull())  # sem item
    .select(
        "snapshot_date",
        "purchase_id",
        "buyer_id",
        "prod_item_id",
        "order_date",
        "release_date",
        "producer_id",
        "subsidiary",
        "transaction_date",
        "snapshot_datetime"
    )
)

df_quarantine_missing_item.write.format("delta") \
    .mode("append") \
    .partitionBy("snapshot_date") \
    .save(QUARANTINE_PATH)

# 2) Métricas DQ do run
df_dq_metrics = (
    df_gvm_snapshot.agg(
        countDistinct("purchase_id").alias("total_purchases_asof"),
        countDistinct(F.when(col("release_date").isNotNull(), col("purchase_id"))).alias("succeded_transactions"),
        F.sum(F.when(col("product_id").isNull() & col("release_date").isNotNull(), 1).otherwise(0)).alias("missing_product_item_cnt"),
        F.sum(F.when(col("subsidiary").isNull(), 1).otherwise(0)).alias("missing_extra_info_cnt"),
        F.sum(F.when(col("subsidiary") == "UNKNOWN", 1).otherwise(0)).alias("unknown_subsidiary_cnt"),
        F.count("*").alias("gvm_rows"),
        spark_sum(col("item_quantity") * col("purchase_value")).alias("gmv_total_value")
    )
    .withColumn("snapshot_date", lit(df_snapshot_date).cast("date"))
    .withColumn("snapshot_datetime", current_timestamp())
)

df_dq_metrics.write.format("delta") \
    .mode("append") \
    .partitionBy("snapshot_date") \
    .save(DQ_PATH)


In [67]:
df_dq_metrics.show(truncate = False)

+--------------------+---------------------+------------------------+----------------------+----------------------+--------+---------------+-------------+-----------------------+
|total_purchases_asof|succeded_transactions|missing_product_item_cnt|missing_extra_info_cnt|unknown_subsidiary_cnt|gvm_rows|gmv_total_value|snapshot_date|snapshot_datetime      |
+--------------------+---------------------+------------------------+----------------------+----------------------+--------+---------------+-------------+-----------------------+
|2                   |2                    |0                       |0                     |0                     |2       |4550.0         |2026-02-26   |2026-02-27 11:31:54.124|
+--------------------+---------------------+------------------------+----------------------+----------------------+--------+---------------+-------------+-----------------------+
